In [0]:
base_path = "/Volumes/workspace/default/autoloader_lab"

landing_path = f"{base_path}/landing"
schema_path = f"{base_path}/schema"
checkpoint_path = f"{base_path}/checkpoint"

print("Landing Path:", landing_path)
print("Schema Path:", schema_path)
print("Checkpoint Path:", checkpoint_path)

In [0]:
# Base location of our Auto Loader lab
base_path = "/Volumes/workspace/default/autoloader_lab"

# Source and schema locations
landing_path = f"{base_path}/landing"
schema_path = f"{base_path}/schema"

# Explicit schema since landing files use .json.txt extension
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

json_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
])

# Create Auto Loader streaming DataFrame
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("pathGlobFilter", "*.json.txt")
    .option("cloudFiles.schemaLocation", schema_path)
    .schema(json_schema)
    .load(landing_path)
)

# Display the incoming streaming data
display(df, checkpointLocation=f"{base_path}/checkpoint_display")

#We are going to read data as a stream, not as a one-time batch.
spark.readStream

In [0]:
#WRITE AUTO LOADER DATA TO BRONZE

base_path = "/Volumes/workspace/default/autoloader_lab"

checkpoint_path = f"{base_path}/checkpoint"

query = (
    df.writeStream
    .format("delta") # Means our Bronze data will be stored as a Delta table.
    .outputMode("append") #add new record
    .option("checkpointLocation", checkpoint_path) #Spark remember the progress of the streaming query.
    .trigger(availableNow=True) #Process all currently available files, then stop
    .toTable("bronze_customers")
)

#Bronze ingestion completed.